# Демо 2 · Три способи оцінити рух: 2D-2D, 3D-3D, 3D-2D
**Заняття 3 · до слайдів «2. Оцінка відносного руху», «Метод 3D-to-2D: PnP VO», «Який метод коли?», «Дрейф: чому одометрії ніколи не достатньо», «ATE проти RPE»**

Один і той самий фронтенд (KLT-трекінг ознак) і три різні оцінювачі руху на тих самих кадрах KITTI 00:

| Метод | Сенсор | Що дає на вході | Розв'язувач |
|---|---|---|---|
| **2D-2D** | лише ліва камера | пари пікселів t−1 → t | Essential matrix (5 точок) + RANSAC, `recoverPose` |
| **3D-3D** | стереопара | 3D-точки в t−1 і в t | Kabsch / SVD + RANSAC |
| **3D-2D** | стереопара | 3D-точки в t−1 і пікселі в t | PnP (EPnP) + RANSAC + LM |

Різниця між траєкторіями — лише від методу, бо точки, кадри й пороги однакові. Далі — порівняння з ground truth: траєкторії, похибка вздовж шляху, проблема масштабу монокуляра і чому 3D-3D «повертає» гірше за PnP. Наприкінці — злиття з (імітованими) колесами та гіроскопом: колеса дають монокуляру масштаб, фільтр Калмана з гіроскопом виправляє курс.

**Дані.** KITTI 00, перші 800 кадрів: `python tools/fetch_kitti.py --no-lidar` з кореня репозиторію (див. `artifacts/kitti/README.md`).

In [ ]:
from pathlib import Path
import time
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

KITTI_ROOT = Path('../../artifacts/kitti')     # сюди кладе дані tools/fetch_kitti.py
SEQUENCE = '00'
N_FRAMES = 800          # ≈ 560 м; у кадрах 500–600 машина повністю зупиняється — це важливо для 2D-2D
N_FEATURES = 1500
MAX_DEPTH_B = 40.0      # стерео-точки далі за 40·B ≈ 21 м відкидаємо (правило ORB-SLAM2)
OUT_DIR = Path('outputs'); OUT_DIR.mkdir(exist_ok=True)

METHODS = ['2D-2D', '3D-3D', '3D-2D']
COLORS = {'GT': '#3b3b3b', '2D-2D': '#eb6834', '3D-3D': '#1baf7a', '3D-2D': '#2a78d6'}

## 1 · Дані: калібрування, пози, кадри

In [ ]:
def load_seq(root, seq):
    sd = Path(root) / 'sequences' / seq
    cal = {}
    for line in (sd / 'calib.txt').read_text().splitlines():
        if ':' in line:
            k, v = line.split(':', 1); cal[k.strip()] = np.array(v.split(), float).reshape(3, 4)
    P0, P1 = cal['P0'], cal['P1']
    K = P0[:, :3]; B = -P1[0, 3] / P1[0, 0]
    pp = Path(root) / 'poses' / f'{seq}.txt'
    gt = np.loadtxt(pp).reshape(-1, 3, 4) if pp.exists() else None
    files = sorted((sd / 'image_0').glob('*.png'))
    return dict(dir=sd, K=K, B=B, gt=gt, n=len(files))


def load_pair(seq, i):
    l = cv2.imread(str(seq['dir'] / 'image_0' / f'{i:06d}.png'), cv2.IMREAD_GRAYSCALE)
    r = cv2.imread(str(seq['dir'] / 'image_1' / f'{i:06d}.png'), cv2.IMREAD_GRAYSCALE)
    return l, r


assert (KITTI_ROOT / 'sequences' / SEQUENCE / 'image_1').exists(), 'KITTI не знайдено: python tools/fetch_kitti.py --no-lidar'
SEQ = load_seq(KITTI_ROOT, SEQUENCE)
N = min(N_FRAMES, SEQ['n'])
print(f"{SEQ['dir']}: кадрів {SEQ['n']}, беремо {N} · f = {SEQ['K'][0, 0]:.1f} px · B = {SEQ['B']:.3f} м")

## 2 · Спільний фронтенд
**Трекінг:** кути Ші-Томасі на лівому кадрі t−1, KLT у кадр t, перевірка вперед-назад < 1 px.

**Розріджене стерео:** для кожної точки шукаємо її у правому кадрі тим самим KLT. Кадри ректифіковані, тож справжня відповідність лежить на тому ж рядку: `|Δy| < 1 px`, диспаритет `d > 0`, глибина `Z = f·B / d`. Це швидше за щільний SGBM на кожен кадр і дає глибину саме там, де є ознаки.

In [ ]:
LK = dict(winSize=(21, 21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))


def track(prev, cur, n_features):
    p0 = cv2.goodFeaturesToTrack(prev, maxCorners=n_features, qualityLevel=0.01, minDistance=7, blockSize=7)
    if p0 is None:
        return np.empty((0, 2), np.float32), np.empty((0, 2), np.float32)
    p1, st1, _ = cv2.calcOpticalFlowPyrLK(prev, cur, p0, None, **LK)
    pb, st2, _ = cv2.calcOpticalFlowPyrLK(cur, prev, p1, None, **LK)
    fb = np.linalg.norm(p0 - pb, axis=2).ravel()
    ok = (st1.ravel() == 1) & (st2.ravel() == 1) & (fb < 1.0)
    return p0.reshape(-1, 2)[ok], p1.reshape(-1, 2)[ok]


def stereo_depth(left, right, pts, K, B, max_depth_B):
    # Глибина кожної точки з розрідженого стерео; NaN, якщо відповідності немає або точка далі за max_depth_B·B
    Z = np.full(len(pts), np.nan)
    if len(pts) == 0:
        return Z
    p = pts.reshape(-1, 1, 2).astype(np.float32)
    pr, st1, _ = cv2.calcOpticalFlowPyrLK(left, right, p, p.copy(), flags=cv2.OPTFLOW_USE_INITIAL_FLOW, **LK)
    pb, st2, _ = cv2.calcOpticalFlowPyrLK(right, left, pr, None, **LK)
    d = (p - pr).reshape(-1, 2)
    fb = np.linalg.norm(p - pb, axis=2).ravel()
    ok = (st1.ravel() == 1) & (st2.ravel() == 1) & (fb < 1.0) & (np.abs(d[:, 1]) < 1.0) & (d[:, 0] > 0.5)
    Z[ok] = K[0, 0] * B / d[ok, 0]
    Z[Z > max_depth_B * B] = np.nan
    return Z


def backproject(pts, Z, K):
    return np.stack([(pts[:, 0] - K[0, 2]) / K[0, 0] * Z, (pts[:, 1] - K[1, 2]) / K[1, 1] * Z, Z], 1)


def to_T(R, t):
    T = np.eye(4); T[:3, :3] = R; T[:3, 3] = np.ravel(t)
    return T

Що бачить кожен метод на одному кадрі: **2D-2D** використовує всі треки, **3D-3D і 3D-2D** — лише ті, для яких стерео дало глибину ближче за 40·B. Далекі точки (небо, дальній кінець вулиці) стерео-методам недоступні, але саме вони найкраще тримають обертання для монокуляра.

In [ ]:
i = min(100, N - 1)
l_prev, r_prev = load_pair(SEQ, i - 1); l_cur, _ = load_pair(SEQ, i)
p0, p1 = track(l_prev, l_cur, N_FEATURES)
has_z = np.isfinite(stereo_depth(l_prev, r_prev, p0, SEQ['K'], SEQ['B'], MAX_DEPTH_B))

fig, ax = plt.subplots(figsize=(14, 4.6))
ax.imshow(l_cur, cmap='gray'); ax.axis('off')
for mask, col, lab in [(~has_z, COLORS['2D-2D'], f'лише 2D (без стерео-глибини): {np.sum(~has_z)}'),
                       (has_z, COLORS['3D-2D'], f'є стерео-глибина < 40·B: {np.sum(has_z)}')]:
    ax.add_collection(LineCollection(np.stack([p0[mask], p1[mask]], 1), colors=col, linewidths=1.2))
    ax.scatter(p1[mask, 0], p1[mask, 1], s=8, c=col, label=lab)
ax.legend(loc='lower right', fontsize=9, framealpha=0.85)
ax.set_title(f'Кадр {i}: треки KLT {i - 1} → {i}')
plt.show()

## 3 · 2D-2D: Essential matrix
`findEssentialMat` (5-точковий алгоритм Ністера + RANSAC) → `recoverPose` (SVD-розклад на 4 варіанти [R|t] і cheirality check). Результат: `X_t = R·X_{t−1} + t`, але **‖t‖ = 1** — масштаб з двох кадрів не відновити.

Якщо медіанний зсув точок < 1 px (машина стоїть), E вироджена — тоді вважаємо, що руху не було.

In [ ]:
def motion_2d2d(p_prev, p_cur, K, ransac_px=1.0, min_flow_px=1.0):
    if len(p_prev) < 8 or np.median(np.linalg.norm(p_cur - p_prev, axis=1)) < min_flow_px:
        return None, 0
    E, mask = cv2.findEssentialMat(p_prev, p_cur, K, cv2.RANSAC, 0.999, ransac_px)
    if E is None or E.shape != (3, 3):
        return None, 0
    n, R, t, mask = cv2.recoverPose(E, p_prev, p_cur, K, mask=mask)
    if n < 8:
        return None, 0
    return to_T(R, t / np.linalg.norm(t)), int(n)

## 4 · 3D-3D: Kabsch / SVD
Для точки, яку видно в стереопарі і в t−1, і в t, маємо дві 3D-позиції: `A` (система камери t−1) і `B` (система камери t). Шукаємо R, t, що мінімізують `Σ‖R·Aᵢ + t − Bᵢ‖²`. Розв'язок у замкненому вигляді через SVD крос-коваріації (Kabsch/Umeyama), а RANSAC на трійках точок відкидає аутлаєри (поріг 0,3 м).

Це та сама задача, яку розв'язує ICP на кожній ітерації, лише з відомими відповідностями замість «найближчої точки».

In [ ]:
def kabsch(A, B):
    # R, t такі, що B ≈ R·A + t (найменші квадрати)
    ca, cb = A.mean(0), B.mean(0)
    U, _, Vt = np.linalg.svd((A - ca).T @ (B - cb))
    D = np.diag([1, 1, np.sign(np.linalg.det(Vt.T @ U.T))])    # захист від віддзеркалення
    R = Vt.T @ D @ U.T
    return R, cb - R @ ca


def motion_3d3d(p_prev, p_cur, Z_prev, Z_cur, K, ransac_m=0.3, iters=200, rng=None):
    ok = np.isfinite(Z_prev) & np.isfinite(Z_cur)
    if ok.sum() < 12:
        return None, 0
    A, B = backproject(p_prev[ok], Z_prev[ok], K), backproject(p_cur[ok], Z_cur[ok], K)
    rng = rng or np.random.default_rng(0)
    best = None
    for _ in range(iters):
        s = rng.choice(len(A), 3, replace=False)
        R, t = kabsch(A[s], B[s])
        inl = np.linalg.norm(A @ R.T + t - B, axis=1) < ransac_m
        if best is None or inl.sum() > best.sum():
            best = inl
    if best.sum() < 12:
        return None, 0
    R, t = kabsch(A[best], B[best])
    return to_T(R, t), int(best.sum())

## 5 · 3D-2D: PnP
3D-точки t−1 з розрідженого стерео, їхні пікселі в t, `solvePnPRansac` (EPnP) і уточнення Левенберга-Марквардта на інлаєрах. На відміну від 3D-3D, похибка міряється **в пікселях** нового кадру, а не в метрах.

In [ ]:
def motion_3d2d(p_prev, p_cur, Z_prev, K, ransac_px=2.0):
    ok = np.isfinite(Z_prev)
    if ok.sum() < 12:
        return None, 0
    X = backproject(p_prev[ok], Z_prev[ok], K)
    good, rvec, tvec, inl = cv2.solvePnPRansac(X, p_cur[ok], K, None, iterationsCount=200, reprojectionError=ransac_px,
                                               confidence=0.999, flags=cv2.SOLVEPNP_EPNP)
    if not good or inl is None or len(inl) < 12:
        return None, 0
    i = inl.ravel()
    rvec, tvec = cv2.solvePnPRefineLM(X[i], p_cur[ok][i], K, None, rvec, tvec)
    return to_T(cv2.Rodrigues(rvec)[0], tvec), len(i)

## 6 · Один прохід — три траєкторії
Для кожної пари кадрів: один трекінг → три оцінки руху. Поза у світі: `T_w,t = T_w,t−1 · T_{t←t−1}⁻¹`. Якщо метод не впорався, беремо модель сталої швидкості (для 2D-2D — «стоїмо»).

Прохід 800 кадрів триває ≈ 2 хв, тому результат кешується в `outputs/`. Щоб перерахувати, видаліть файл кешу.

In [ ]:
def run_all(seq, n_frames, n_features=N_FEATURES, max_depth_B=MAX_DEPTH_B):
    K, B = seq['K'], seq['B']
    rng = np.random.default_rng(1)
    T0 = np.eye(4)
    if seq['gt'] is not None:
        T0[:3, :] = seq['gt'][0]
    T_w = {m: T0.copy() for m in METHODS}
    out = {m: dict(poses=[T0.copy()], rel=[], inliers=[], sec=0.0, fails=0) for m in METHODS}
    last = {m: np.eye(4) for m in METHODS}
    l_prev, r_prev = load_pair(seq, 0)
    for i in range(1, n_frames):
        l, r = load_pair(seq, i)
        t0 = time.perf_counter(); p0, p1 = track(l_prev, l, n_features); t_track = time.perf_counter() - t0

        t0 = time.perf_counter(); est = {'2D-2D': motion_2d2d(p0, p1, K)}; sec = {'2D-2D': time.perf_counter() - t0}
        t0 = time.perf_counter(); Z0 = stereo_depth(l_prev, r_prev, p0, K, B, max_depth_B); t_z0 = time.perf_counter() - t0
        t0 = time.perf_counter(); est['3D-2D'] = motion_3d2d(p0, p1, Z0, K); sec['3D-2D'] = t_z0 + time.perf_counter() - t0
        t0 = time.perf_counter(); Z1 = stereo_depth(l, r, p1, K, B, max_depth_B)
        est['3D-3D'] = motion_3d3d(p0, p1, Z0, Z1, K, rng=rng); sec['3D-3D'] = t_z0 + time.perf_counter() - t0

        for m, (T, n_inl) in est.items():
            o = out[m]; o['sec'] += t_track + sec[m]
            if T is None:
                o['fails'] += 1
                T = np.eye(4) if m == '2D-2D' else last[m]
            last[m] = T
            T_w[m] = T_w[m] @ np.linalg.inv(T)
            o['poses'].append(T_w[m].copy()); o['rel'].append(T); o['inliers'].append(n_inl)
        l_prev, r_prev = l, r
        if i % 100 == 0:
            print(f'  {i}/{n_frames - 1}')
    return {m: dict(poses=np.array(o['poses']), rel=np.array(o['rel']), inliers=np.array(o['inliers']),
                    ms=1000 * o['sec'] / (n_frames - 1), fails=o['fails']) for m, o in out.items()}


CACHE = OUT_DIR / f'vo_compare_{SEQUENCE}_{N}.npz'
if CACHE.exists():
    z = np.load(CACHE)
    RES = {m: dict(poses=z[f'{m}/poses'], rel=z[f'{m}/rel'], inliers=z[f'{m}/inliers'],
                   ms=float(z[f'{m}/ms']), fails=int(z[f'{m}/fails'])) for m in METHODS}
    print('завантажено з кешу', CACHE)
else:
    t0 = time.time()
    RES = run_all(SEQ, N)
    np.savez(CACHE, **{f'{m}/{k}': v for m, r in RES.items() for k, v in r.items()})
    print(f'{time.time() - t0:.0f} с · збережено {CACHE}')

for m in METHODS:                                   # KITTI-формат для lesson03_demo1_kitti_rerun.py --est-poses
    full = np.repeat(RES[m]['poses'][-1:, :3, :4], SEQ['n'], 0); full[:N] = RES[m]['poses'][:, :3, :4]
    np.savetxt(OUT_DIR / f"vo_{m.replace('-', '').lower()}_{SEQUENCE}.txt", full.reshape(-1, 12))

## 7 · Вирівнювання з ground truth і метрики
- **Стерео-методи** стартують з GT-пози кадру 0 і порівнюються напряму, без жодного вирівнювання: масштаб у них метричний.
- **2D-2D** має невідомий масштаб, тому показуємо три версії:
  - *сира* — кожен крок довжиною 1 «одиниця»;
  - *Sim(3)* — один глобальний масштаб + поворот + зсув, підібрані Умеямою за всією GT-траєкторією заднім числом (так монокуляр оцінюють у статтях);
  - *масштаб із GT* — довжина кожного кроку взята з GT («чит», якого в польоті немає; на практиці його замінюють IMU, висотомір або відомий об'єкт).

**Метрики** (слайд «ATE проти RPE»): ATE (RMSE позиції), кінцевий дрейф у % пройденого шляху, KITTI-подібна похибка на відрізках 100 м (трансляція в %, обертання в °/100 м).

In [ ]:
GT = np.tile(np.eye(4), (N, 1, 1)); GT[:, :3, :] = SEQ['gt'][:N]
GT_REL = np.array([np.linalg.inv(GT[k + 1]) @ GT[k] for k in range(N - 1)])     # X_t = T·X_{t−1}, як у оцінювачів
gt_xyz = GT[:, :3, 3]
dist = np.r_[0, np.cumsum(np.linalg.norm(np.diff(gt_xyz, axis=0), axis=1))]


def chain(rel, T0):
    P = [T0]
    for T in rel:
        P.append(P[-1] @ np.linalg.inv(T))
    return np.array(P)


def umeyama(src, dst, with_scale=True):
    # s, R, t: dst ≈ s·R·src + t
    mu_s, mu_d = src.mean(0), dst.mean(0)
    S, D = src - mu_s, dst - mu_d
    U, sig, Vt = np.linalg.svd(D.T @ S / len(src))
    F = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        F[2, 2] = -1
    R = U @ F @ Vt
    s = np.trace(np.diag(sig) @ F) / S.var(0).sum() if with_scale else 1.0
    return s, R, mu_d - s * R @ mu_s


def apply_sim3(P, s, R, t):
    Q = P.copy(); Q[:, :3, :3] = R @ P[:, :3, :3]; Q[:, :3, 3] = s * P[:, :3, 3] @ R.T + t
    return Q


def rot_deg(R):
    return np.degrees(np.arccos(np.clip((np.trace(R) - 1) / 2, -1, 1)))


def metrics(P, seg=100.0):
    err = np.linalg.norm(P[:, :3, 3] - gt_xyz, axis=1)
    t_seg, r_seg = [], []
    for i in range(0, N, 5):
        j = np.searchsorted(dist, dist[i] + seg)
        if j >= N:
            break
        E = np.linalg.inv(np.linalg.inv(GT[i]) @ GT[j]) @ (np.linalg.inv(P[i]) @ P[j])
        t_seg.append(np.linalg.norm(E[:3, 3]) / seg * 100); r_seg.append(rot_deg(E[:3, :3]) / seg * 100)
    return dict(err=err, ate=np.sqrt(np.mean(err ** 2)), end=100 * err[-1] / dist[-1],
                t_seg=np.mean(t_seg) if t_seg else np.nan, r_seg=np.mean(r_seg) if r_seg else np.nan)


mono_raw = RES['2D-2D']['poses']
mono_gt_scale = chain([to_T(T[:3, :3], T[:3, 3] * np.linalg.norm(G[:3, 3])) for T, G in zip(RES['2D-2D']['rel'], GT_REL)], GT[0])
s_mono, R_mono, t_mono = umeyama(mono_raw[:, :3, 3], gt_xyz)
mono_sim3 = apply_sim3(mono_raw, s_mono, R_mono, t_mono)

TRAJ = {'2D-2D': mono_gt_scale, '3D-3D': RES['3D-3D']['poses'], '3D-2D': RES['3D-2D']['poses']}
LABEL = {'2D-2D': '2D-2D · масштаб із GT', '3D-3D': '3D-3D · стерео + Kabsch', '3D-2D': '3D-2D · стерео + PnP'}
MET = {m: metrics(P) for m, P in TRAJ.items()}
MET_SIM3 = metrics(mono_sim3)
ROT_STEP = {m: np.array([rot_deg(T[:3, :3] @ G[:3, :3].T) for T, G in zip(RES[m]['rel'], GT_REL)]) for m in METHODS}

print(f"шлях {dist[-1]:.0f} м, {N} кадрів · глобальний масштаб монокуляра (Sim(3)): s = {s_mono:.3f} м/од.\n")
hdr = f"{'метод':<26}{'ATE, м':>8}{'дрейф, %':>10}{'t/100м, %':>11}{'r, °/100м':>11}{'Δr/кадр, °':>12}{'інлаєри':>9}{'мс/кадр':>9}{'без оцінки':>12}"
print(hdr); print('─' * len(hdr))
for m in METHODS:
    k = MET[m]; r = RES[m]
    print(f"{LABEL[m]:<26}{k['ate']:>8.2f}{k['end']:>10.2f}{k['t_seg']:>11.2f}{k['r_seg']:>11.3f}{ROT_STEP[m].mean():>12.3f}"
          f"{r['inliers'].mean():>9.0f}{r['ms']:>9.1f}{r['fails']:>12d}")
k = MET_SIM3
print(f"{'2D-2D · Sim(3)':<26}{k['ate']:>8.2f}{k['end']:>10.2f}{k['t_seg']:>11.2f}{k['r_seg']:>11.3f}{'':>12}{'':>9}{'':>9}{'':>12}")
print()
print('«без оцінки» для 2D-2D — переважно кадри зупинки (зсув < 1 px), там вважаємо, що руху не було.')

## 8 · Траєкторії поруч із ground truth
Вид зверху (x — вбік, z — вперед). Усі три методи бачили однакові пікселі.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 8.5))
ax.plot(gt_xyz[:, 0], gt_xyz[:, 2], color=COLORS['GT'], ls='--', lw=2, label='GT (KITTI)')
for m in METHODS:
    xyz = TRAJ[m][:, :3, 3]
    ax.plot(xyz[:, 0], xyz[:, 2], color=COLORS[m], lw=2, label=f"{LABEL[m]} · ATE {MET[m]['ate']:.1f} м")
    ax.plot(*xyz[-1, [0, 2]], 'o', ms=8, color=COLORS[m], mec='white', mew=2)
ax.plot(*gt_xyz[0, [0, 2]], 's', ms=9, color=COLORS['GT'], mec='white', mew=2)
ax.annotate('старт', gt_xyz[0, [0, 2]], xytext=(8, -14), textcoords='offset points', color='#555')
ax.plot(*gt_xyz[-1, [0, 2]], 'o', ms=8, color=COLORS['GT'], mec='white', mew=2)
ax.set_aspect('equal', adjustable='datalim'); ax.grid(alpha=0.25)
ax.set_xlabel('x, м'); ax.set_ylabel('z, м'); ax.legend(loc='best', fontsize=9)
ax.set_title(f'KITTI {SEQUENCE}, перші {N} кадрів ({dist[-1]:.0f} м)')
plt.show()

Кожен метод окремо, лінія розфарбована за похибкою позиції (одна шкала для всіх трьох). Видно, **де** саме метод починає розходитися з GT — переважно після поворотів.

In [ ]:
vmax = max(np.percentile(MET[m]['err'], 99) for m in METHODS)
norm = plt.Normalize(0, vmax)
cmap = plt.matplotlib.colors.ListedColormap(plt.cm.Reds(np.linspace(0.3, 1, 256)))   # без білого краю — мала похибка теж видна
allxz = np.concatenate([gt_xyz[:, [0, 2]]] + [TRAJ[m][:, :3, 3][:, [0, 2]] for m in METHODS])
lo, hi = allxz.min(0) - 10, allxz.max(0) + 10
fig, axes = plt.subplots(1, 3, figsize=(17, 6.6), layout='constrained')
for ax, m in zip(axes, METHODS):
    xz = TRAJ[m][:, :3, 3][:, [0, 2]]
    ax.plot(gt_xyz[:, 0], gt_xyz[:, 2], color=COLORS['GT'], ls='--', lw=1.2, alpha=0.7)
    lc = LineCollection(np.stack([xz[:-1], xz[1:]], 1), cmap=cmap, norm=norm, linewidths=3)
    lc.set_array(MET[m]['err'][1:]); ax.add_collection(lc)
    ax.set_title(f"{LABEL[m]}\nATE {MET[m]['ate']:.1f} м · дрейф {MET[m]['end']:.1f} %", color=COLORS[m], fontsize=11)
    ax.set_xlim(lo[0], hi[0]); ax.set_ylim(lo[1], hi[1]); ax.set_aspect('equal'); ax.grid(alpha=0.25); ax.set_xlabel('x, м')
axes[0].set_ylabel('z, м')
fig.colorbar(lc, ax=axes, shrink=0.8, label='похибка позиції, м')
plt.show()

Похибка як функція пройденого шляху. Накопичення монотонне — це і є дрейф з слайда «Дрейф: чому одометрії ніколи не достатньо».

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
for m in METHODS:
    ax.plot(dist, MET[m]['err'], color=COLORS[m], lw=2, label=LABEL[m])
    ax.annotate(f"{MET[m]['err'][-1]:.1f} м", (dist[-1], MET[m]['err'][-1]), xytext=(6, 0), textcoords='offset points',
                va='center', fontsize=9, color='#333')
ax.set_xlabel('пройдено, м'); ax.set_ylabel('похибка позиції, м'); ax.grid(alpha=0.25); ax.legend(loc='upper left')
ax.set_xlim(0, dist[-1] * 1.08)
plt.show()

## 9 · Монокуляр: форма правильна, масштаб — ні
1. **Сира** траєкторія 2D-2D у власних «одиницях»: форма впізнавана, але метрів тут немає.
2. **Довжина кроку**: стерео-методи міряють швидкість у метрах, монокуляр завжди повертає 1. Сіра зона — зупинка на світлофорі: там 2D-2D просто не рухається (ми це заклали в `min_flow_px`).
3. **Що дає масштаб**: один глобальний масштаб Sim(3) не врятує, бо машина то гальмує, то розганяється; масштаб на кожному кроці (тут — «чит» із GT) майже повертає траєкторію.

In [ ]:
step_gt = np.linalg.norm(GT_REL[:, :3, 3], axis=1)
stop = step_gt < 0.05
fig, ax = plt.subplots(1, 3, figsize=(18, 5.6), gridspec_kw=dict(width_ratios=[1, 1.5, 1]), layout='constrained')

xyz = mono_raw[:, :3, 3] - mono_raw[0, :3, 3]
ax[0].plot(xyz[:, 0], xyz[:, 2], color=COLORS['2D-2D'], lw=2)
ax[0].set_aspect('equal', adjustable='datalim'); ax[0].grid(alpha=0.25)
ax[0].set_xlabel('x, од.'); ax[0].set_ylabel('z, од.'); ax[0].set_title('2D-2D сира: одиниці невідомі')

fr = np.arange(1, N)
for lo, hi in zip(*[np.flatnonzero(np.diff(np.r_[0, stop.astype(int), 0]) == s) for s in (1, -1)]):
    ax[1].axvspan(lo + 1, hi + 1, color='#d0d0d0', alpha=0.6, lw=0)
ax[1].plot(fr, step_gt, color=COLORS['GT'], lw=2, ls='--', label='GT')
for m in ['3D-3D', '3D-2D']:
    ax[1].plot(fr, np.linalg.norm(RES[m]['rel'][:, :3, 3], axis=1), color=COLORS[m], lw=1, alpha=0.9, label=m)
ax[1].plot(fr, np.linalg.norm(RES['2D-2D']['rel'][:, :3, 3], axis=1), color=COLORS['2D-2D'], lw=2, label='2D-2D (од./кадр)')
ax[1].set_ylim(0, 1.3); ax[1].grid(alpha=0.25); ax[1].legend(loc='lower left', fontsize=9, ncol=2)
ax[1].set_xlabel('кадр'); ax[1].set_ylabel('довжина кроку, м/кадр'); ax[1].set_title('Швидкість: стерео бачить метри, монокуляр — ні')

ax[2].plot(gt_xyz[:, 0], gt_xyz[:, 2], color=COLORS['GT'], ls='--', lw=2, label='GT')
ax[2].plot(mono_sim3[:, 0, 3], mono_sim3[:, 2, 3], color=COLORS['2D-2D'], lw=1.5, ls=':',
           label=f"один масштаб Sim(3) · ATE {MET_SIM3['ate']:.1f} м")
ax[2].plot(mono_gt_scale[:, 0, 3], mono_gt_scale[:, 2, 3], color=COLORS['2D-2D'], lw=2,
           label=f"масштаб на кожен крок · ATE {MET['2D-2D']['ate']:.1f} м")
ax[2].set_aspect('equal', adjustable='datalim'); ax[2].grid(alpha=0.25); ax[2].legend(loc='best', fontsize=9)
ax[2].set_xlabel('x, м'); ax[2].set_ylabel('z, м'); ax[2].set_title('2D-2D з масштабом')
plt.show()

## 10 · Чому 3D-3D гірше за PnP, хоча вхідні дані ті самі
Подивимось на обертання окремо. Ліворуч — **знакова** похибка курсу (поворот навколо вертикальної осі камери) на кожному кадрі: у PnP вона симетрична навколо нуля, а у 3D-3D зсунута в один бік. Шум усереднюється, а зміщення — ні: +0,1° на кадр дає десятки градусів за кількасот кадрів, і траєкторія «завертає».

Звідки зміщення. Коли машина їде вперед, для близьких точок поворот на кут δ і бічний зсув камери дають майже однакову зміну 3D-координат — у 3D-3D ці дві невідомі погано розділяються, і будь-яка систематична похибка стерео (наприклад, на похилих поверхнях фасадів і дороги) йде в курс. PnP працює з напрямками на точки в пікселях: поворот камери зсуває **всі** точки однаково незалежно від глибини, тож обертання він фіксує набагато жорсткіше.

Праворуч — накопичена похибка орієнтації вздовж шляху.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 4.8), layout='constrained')
YAW_STEP = {m: np.array([np.degrees(cv2.Rodrigues(T[:3, :3] @ G[:3, :3].T)[0][1, 0]) for T, G in zip(RES[m]['rel'], GT_REL)])
            for m in METHODS}
lim = np.percentile(np.abs(np.concatenate(list(YAW_STEP.values()))), 99)
bins = np.linspace(-lim, lim, 60)
for m in METHODS:
    ax[0].hist(YAW_STEP[m], bins=bins, histtype='step', lw=2, color=COLORS[m], label=f'{m} · середня {YAW_STEP[m].mean():+.3f}°')
ax[0].axvline(0, color=COLORS['GT'], lw=1, ls='--')
ax[0].set_xlabel('похибка курсу за кадр, ° (зі знаком)'); ax[0].set_ylabel('кадрів'); ax[0].legend(); ax[0].grid(alpha=0.25)
ax[0].set_title('Курс кадр-до-кадру: шум чи зміщення?')
for m in METHODS:
    acc = [rot_deg(P[:3, :3].T @ G[:3, :3]) for P, G in zip(TRAJ[m], GT)]
    ax[1].plot(dist, acc, color=COLORS[m], lw=2, label=LABEL[m])
ax[1].set_xlabel('пройдено, м'); ax[1].set_ylabel('похибка орієнтації, °'); ax[1].legend(loc='upper left'); ax[1].grid(alpha=0.25)
ax[1].set_title('Накопичена похибка орієнтації')
plt.show()

## 11 · Те саме в Rerun
Три оцінки й GT в одному 3D-вікні, синхронно з кадром камери і похибками на шкалі часу. Перемотуйте таймлайн `frame`.

Окремий метод можна подивитися і через demo1 разом зі сканом LiDAR:
`python lesson03_demo1_kitti_rerun.py --frames 800 --est-poses outputs/vo_3d2d_00.txt`

In [ ]:
def show_in_rerun(step=2, save=None):
    import rerun as rr
    import rerun.blueprint as rrb
    rr.init('cv_deftech_l3_vo_methods', spawn=save is None)
    if save:
        rr.save(str(save))
    rr.send_blueprint(rrb.Blueprint(rrb.Horizontal(
        rrb.Spatial3DView(name='Траєкторії', origin='/world'),
        rrb.Vertical(rrb.Spatial2DView(name='Ліва камера', origin='/world/gt_cam/image'),
                     rrb.TimeSeriesView(name='Похибка позиції, м', origin='/error')),
        column_shares=[3, 2])))
    rgb = lambda h: [int(h[i:i + 2], 16) for i in (1, 3, 5)]
    rr.log('world', rr.ViewCoordinates.RDF, static=True)
    rr.log('world/gt_full', rr.LineStrips3D([gt_xyz], colors=[[180, 180, 180]], radii=0.05), static=True)
    K = SEQ['K']; h, w = load_pair(SEQ, 0)[0].shape
    rr.log('world/gt_cam/image', rr.Pinhole(image_from_camera=K, width=w, height=h, image_plane_distance=2.0), static=True)
    for m in METHODS:
        rr.log(f'error/{m}', rr.SeriesLines(colors=[rgb(COLORS[m])], names=[LABEL[m]]), static=True)
    for k in range(0, N, step):
        rr.set_time('frame', sequence=k)
        rr.log('world/gt_cam', rr.Transform3D(translation=GT[k, :3, 3], mat3x3=GT[k, :3, :3]))
        rr.log('world/gt_cam/image', rr.Image(load_pair(SEQ, k)[0]))
        rr.log('world/gt', rr.LineStrips3D([gt_xyz[:k + 1]], colors=[rgb(COLORS['GT'])], radii=0.15))
        for m in METHODS:
            rr.log(f'world/{m}', rr.LineStrips3D([TRAJ[m][:k + 1, :3, 3]], colors=[rgb(COLORS[m])], radii=0.15))
            rr.log(f'error/{m}', rr.Scalars(float(MET[m]['err'][k])))


show_in_rerun()          # або show_in_rerun(step=5, save=OUT_DIR / 'vo_methods.rrd') — файл для студентів (з кадрами важить десятки МБ)

## 12 · Злиття з колесами та гіроскопом
VO сама по собі має дві слабкі місця, які ми щойно бачили: **масштаб** (2D-2D) і **курс** (особливо 3D-3D). Наземна платформа майже завжди має ще два дешеві сенсори: енкодери коліс (швидкість) і MEMS-гіроскоп (швидкість повороту). Подивимось, що вони дають.

**Чому імітація.** В odometry-наборі KITTI немає ні коліс, ні IMU. У raw-наборі є OXTS (`vf`, `wz`), але це вихід дорогої GPS/INS RT3003 — майже ідеальні дані, зовсім не схожі на колеса чи гіроскоп польотного контролера. Тому генеруємо показання з GT із реалістичною моделлю похибок і чесно називаємо це імітацією:

| Сенсор | Модель | Типові значення |
|---|---|---|
| Колеса | `v = (1 + ε)·v_GT + шум`, на стоянці рівно 0 | ε = 1–3 % (тиск, знос шин), шум 0,1 м/с |
| Гіроскоп | `ω = ω_GT + b + шум` | некалібрований MEMS: b = 0,1–0,5 °/с; шум 0,1 °/с |

Розглядаємо лише курс (поворот навколо вертикалі): KITTI 00 майже пласка, тож 2D-модель руху достатня.

In [ ]:
WHEEL_SCALE_ERR = 0.02    # колесо «бреше» на +2 %
WHEEL_NOISE = 0.1         # м/с
GYRO_BIAS_DPS = 0.2       # °/с — некалібрований MEMS
GYRO_NOISE_DPS = 0.1      # °/с на відлік
CALIB_S = 0.0             # скільки секунд гіроскоп калібрувався нерухомим перед стартом
VO_NOISE_DEG = 0.1        # σ приросту курсу з VO за кадр (з гістограми в розділі 10)
BIAS_RW_DPS = 0.001       # повільний дрейф самого зсуву гіроскопа, °/с/√с

TIMES = np.loadtxt(SEQ['dir'] / 'times.txt')[:N]
DT = np.diff(TIMES)


def yaw_of(R):
    # курс: куди дивиться вісь z камери в площині x-z
    return np.arctan2(R[0, 2], R[2, 2])


def step_yaw(T):
    # приріст курсу за крок для T = T_{t←t−1} (поза оновлюється як T_w,t = T_w,t−1 · T⁻¹)
    return yaw_of(np.linalg.inv(T)[:3, :3])


def Ry(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def wrap(a):
    return (a + np.pi) % (2 * np.pi) - np.pi


GT_DYAW = np.array([step_yaw(T) for T in GT_REL])
GT_STEP = np.linalg.norm(GT_REL[:, :3, 3], axis=1)
GT_YAW = np.array([yaw_of(G[:3, :3]) for G in GT])


def simulate_sensors(wheel_scale_err=WHEEL_SCALE_ERR, wheel_noise=WHEEL_NOISE, gyro_bias_dps=GYRO_BIAS_DPS,
                     gyro_noise_dps=GYRO_NOISE_DPS, calib_s=CALIB_S, seed=0):
    rng = np.random.default_rng(seed)
    moving = GT_STEP > 0.01
    v = np.maximum((1 + wheel_scale_err) * GT_STEP / DT + rng.normal(0, wheel_noise, len(DT)) * moving, 0)
    b, sw = np.radians(gyro_bias_dps), np.radians(gyro_noise_dps)
    w = GT_DYAW / DT + b + rng.normal(0, sw, len(DT))
    n_cal = int(calib_s / DT.mean())
    if n_cal:                                  # стоїмо calib_s секунд: середнє показів гіроскопа і є оцінка зсуву
        b0, var_b0 = np.mean(b + rng.normal(0, sw, n_cal)), sw ** 2 / n_cal
    else:                                      # без калібрування: знаємо лише паспортний діапазон ±0,5 °/с
        b0, var_b0 = 0.0, np.radians(0.5) ** 2
    return dict(v=v, w=w, b0=b0, var_b0=var_b0, still=v < 0.05)


SENS = simulate_sensors()
print(f"колеса: {SENS['v'].mean():.2f} м/с у середньому, нерухомих кадрів {SENS['still'].sum()} · "
      f"гіроскоп: оцінка зсуву на старті {np.degrees(SENS['b0']):.3f} °/с (справжній {GYRO_BIAS_DPS} °/с)")

### 12.1 · Колеса виправляють масштаб монокуляра
Напрямок кроку беремо з Essential matrix (‖t‖ = 1), довжину — `v·Δt` з коліс. Це та сама схема, що й «масштаб із GT» з розділу 9, лише замість чита — реальний сенсор. Похибка радіуса колеса ε рівномірно розтягує траєкторію: форма не страждає, а дрейф ≈ ε.

In [ ]:
def vo_step_dirs(rel):
    # напрямок зсуву камери в системі t−1; для «стоїмо» (T = I) — вперед, але довжина тоді 0
    d = np.array([-T[:3, :3].T @ T[:3, 3] for T in rel])
    n = np.linalg.norm(d, axis=1, keepdims=True)
    return np.where(n > 1e-9, d / np.maximum(n, 1e-9), [0, 0, 1.0])


def mono_with_wheels(v):
    P = [GT[0]]
    for T, d, l in zip(RES['2D-2D']['rel'], vo_step_dirs(RES['2D-2D']['rel']), v * DT):
        M = np.linalg.inv(T); M[:3, 3] = d * l
        P.append(P[-1] @ M)
    return np.array(P)


fig, ax = plt.subplots(figsize=(8.5, 8.5))
ax.plot(gt_xyz[:, 0], gt_xyz[:, 2], color=COLORS['GT'], ls='--', lw=2, label='GT')
ax.plot(mono_sim3[:, 0, 3], mono_sim3[:, 2, 3], color=COLORS['2D-2D'], lw=1.5, ls=':',
        label=f"моно, один масштаб Sim(3) з GT · ATE {MET_SIM3['ate']:.1f} м")
for eps, ls in [(0.0, '-'), (WHEEL_SCALE_ERR, '-.'), (-0.05, '--')]:
    P = mono_with_wheels(simulate_sensors(wheel_scale_err=eps)['v']); k = metrics(P)
    ax.plot(P[:, 0, 3], P[:, 2, 3], color=COLORS['2D-2D'], lw=2 if eps == 0 else 1.5, ls=ls,
            label=f"моно + колеса, ε = {eps:+.0%} · ATE {k['ate']:.1f} м · дрейф {k['end']:.1f} %")
ax.set_aspect('equal', adjustable='datalim'); ax.grid(alpha=0.25); ax.legend(loc='best', fontsize=9)
ax.set_xlabel('x, м'); ax.set_ylabel('z, м'); ax.set_title('2D-2D + колеса: масштаб із реального сенсора, без GT')
plt.show()

### 12.2 · Курс: фільтр Калмана «гіроскоп + VO»
Гіроскоп і VO обидва **відносні**: жоден не міряє абсолютний курс. Тому тут не потрібен EKF — достатньо лінійного фільтра Калмана з трьома станами:

- `ψ_k` — курс зараз, `ψ_{k−1}` — його копія з попереднього кроку («клон»), `b` — зсув гіроскопа;
- **передбачення:** `ψ_k = ψ_{k−1} + (ω − b)·Δt` — гіроскоп інтегрується з поправкою на оцінений зсув;
- **вимірювання VO:** приріст курсу `Δψ_VO = ψ_k − ψ_{k−1}` (тому й потрібен клон);
- **ZARU** (zero angular rate update): якщо колеса показують 0, машина стоїть і справжня швидкість повороту 0 — тоді гіроскоп напряму міряє власний зсув: `ω = b`.

Камера в цьому фільтрі не «виправляє курс» напряму — вона дає змогу **оцінити зсув гіроскопа**. Справжній EKF з'явиться тоді, коли додасться абсолютне вимірювання (GPS, орієнтири карти, замикання петлі в demo3).

In [ ]:
def heading_kf(w, still, vo_dyaw, vo_ok, b0, var_b0, gyro_noise_dps=GYRO_NOISE_DPS, vo_noise_deg=VO_NOISE_DEG,
               bias_rw_dps=BIAS_RW_DPS):
    x = np.array([0.0, 0.0, b0])                               # [ψ_k, ψ_{k−1}, b]
    P = np.diag([1e-12, 1e-12, var_b0])
    sw, svo, sbw = np.radians(gyro_noise_dps), np.radians(vo_noise_deg), np.radians(bias_rw_dps)
    psi, bias = [x[0]], [x[2]]
    for k, dt in enumerate(DT):
        x[1] = x[0]; P[1, :] = P[0, :]; P[:, 1] = P[:, 0]       # клон попереднього курсу
        F = np.array([[1.0, 0, -dt], [0, 1, 0], [0, 0, 1]])
        x = F @ x + np.array([w[k] * dt, 0, 0])
        P = F @ P @ F.T + np.diag([(sw * dt) ** 2, 0, sbw ** 2 * dt])
        if still[k]:
            H, z, r = np.array([0, 0, 1.0]), w[k], sw ** 2         # ZARU: стоїмо → гіроскоп показує лише зсув
        elif vo_ok[k]:
            H, z, r = np.array([1.0, -1.0, 0]), vo_dyaw[k], svo ** 2
        else:
            psi.append(x[0]); bias.append(x[2]); continue
        S = H @ P @ H + r
        K = P @ H / S
        x = x + K * (z - H @ x)
        P = (np.eye(3) - np.outer(K, H)) @ P
        psi.append(x[0]); bias.append(x[2])
    return np.array(psi), np.array(bias)

### 12.3 · Усі комбінації на одних даних
Для кожної VO беремо її приріст курсу як вимірювання у фільтрі, а довжину й напрямок кроку — з неї самої (для 2D-2D довжина з коліс). Окремо — класична одометрія наземного робота без камери: колеса + гіроскоп із тим зсувом, який вдалося оцінити на старті.

In [ ]:
def planar_chain(psi, dirs, lengths):
    # поза з курсу ψ, напрямку кроку в системі камери t−1 і довжини кроку (крен і тангаж ігноруємо)
    R0 = GT[0, :3, :3]
    P = [GT[0].copy()]
    for k in range(len(lengths)):
        T = np.eye(4); T[:3, :3] = R0 @ Ry(psi[k + 1])
        T[:3, 3] = P[-1][:3, 3] + R0 @ Ry(psi[k]) @ dirs[k] * lengths[k]
        P.append(T)
    return np.array(P)


FUSION_STYLE = {
    'колеса + гіроскоп':       dict(color='#4a3aa7', ls='-', lw=2),
    '2D-2D + колеса':          dict(color=COLORS['2D-2D'], ls=':', lw=1.6),
    '2D-2D + колеса + гіроскоп': dict(color=COLORS['2D-2D'], ls='-', lw=2.2),
    '3D-2D':                   dict(color=COLORS['3D-2D'], ls=':', lw=1.6),
    '3D-2D + гіроскоп':        dict(color=COLORS['3D-2D'], ls='-', lw=2.2),
    '3D-3D':                   dict(color=COLORS['3D-3D'], ls=':', lw=1.6),
    '3D-3D + гіроскоп':        dict(color=COLORS['3D-3D'], ls='-', lw=2.2),
}


def fusion_experiment(vo_noise_deg=VO_NOISE_DEG, bias_rw_dps=BIAS_RW_DPS, **sensor_kw):
    s = simulate_sensors(**sensor_kw)
    L = s['v'] * DT
    traj, bias = {}, {}
    traj['колеса + гіроскоп'] = planar_chain(np.r_[0, np.cumsum((s['w'] - s['b0']) * DT)], np.tile([0, 0, 1.0], (len(DT), 1)), L)
    traj['2D-2D + колеса'] = mono_with_wheels(s['v'])
    for m in METHODS:
        rel = RES[m]['rel']
        ok = np.array([not np.allclose(T, np.eye(4)) for T in rel])     # 2D-2D повертає I, коли не бачить руху
        psi, bias[m] = heading_kf(s['w'], s['still'], np.array([step_yaw(T) for T in rel]), ok, s['b0'], s['var_b0'],
                                  sensor_kw.get('gyro_noise_dps', GYRO_NOISE_DPS), vo_noise_deg, bias_rw_dps)
        if m == '2D-2D':
            traj['2D-2D + колеса + гіроскоп'] = planar_chain(psi, vo_step_dirs(rel), L)
        else:
            steps = np.array([-T[:3, :3].T @ T[:3, 3] for T in rel]); n = np.linalg.norm(steps, axis=1)
            traj[m] = RES[m]['poses']
            traj[f'{m} + гіроскоп'] = planar_chain(psi, steps / np.maximum(n, 1e-9)[:, None], n)
    yaw_err = {k: np.degrees(np.abs(wrap(np.array([yaw_of(T[:3, :3]) for T in P]) - GT_YAW))) for k, P in traj.items()}
    met = {k: metrics(P) for k, P in traj.items()}
    stats = {k: dict(ate=met[k]['ate'], end=met[k]['end'], yaw_end=yaw_err[k][-1]) for k in traj}
    return dict(gt_xyz=gt_xyz, dist=dist, xyz={k: P[:, :3, 3] for k, P in traj.items()}, yaw_err=yaw_err, stats=stats,
                bias=bias, true_bias_dps=sensor_kw.get('gyro_bias_dps', GYRO_BIAS_DPS), style=FUSION_STYLE)


FUS = fusion_experiment()
print(f"гіроскоп: зсув {GYRO_BIAS_DPS} °/с, калібрування {CALIB_S:.0f} с · колеса: ε = {WHEEL_SCALE_ERR:+.0%}\n")
print(f"{'комбінація':<28}{'ATE, м':>8}{'дрейф, %':>10}{'курс у кінці, °':>17}")
print('─' * 63)
for k, st in FUS['stats'].items():
    print(f"{k:<28}{st['ate']:>8.2f}{st['end']:>10.2f}{st['yaw_end']:>17.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 8.5))
ax.plot(gt_xyz[:, 0], gt_xyz[:, 2], color=COLORS['GT'], ls='--', lw=2, label='GT')
for k in ['колеса + гіроскоп', '2D-2D + колеса', '2D-2D + колеса + гіроскоп']:
    xyz = FUS['xyz'][k]
    ax.plot(xyz[:, 0], xyz[:, 2], **FUS['style'][k], label=f"{k} · ATE {FUS['stats'][k]['ate']:.1f} м")
ax.set_aspect('equal', adjustable='datalim'); ax.grid(alpha=0.25); ax.legend(loc='best', fontsize=9)
ax.set_xlabel('x, м'); ax.set_ylabel('z, м'); ax.set_title('Моно-камера + колеса + гіроскоп')
plt.show()

Курс окремо для кожної VO: пунктир — лише VO, суцільна — VO + гіроскоп через KF, фіолетова — лише гіроскоп (зі зсувом, оціненим на старті). Шкали різні: у 3D-3D похибка на порядок більша.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), layout='constrained')
for ax, (vo, fused) in zip(axes, [('2D-2D + колеса', '2D-2D + колеса + гіроскоп'), ('3D-2D', '3D-2D + гіроскоп'),
                                  ('3D-3D', '3D-3D + гіроскоп')]):
    for k in ['колеса + гіроскоп', vo, fused]:
        ax.plot(dist, FUS['yaw_err'][k], **FUS['style'][k], label=f"{k} · {FUS['yaw_err'][k][-1]:.1f}°")
    ax.set_ylim(0, None); ax.grid(alpha=0.25); ax.legend(loc='upper left', fontsize=9)
    ax.set_xlabel('пройдено, м'); ax.set_title(fused.split(' +')[0], color=FUS['style'][fused]['color'])
axes[0].set_ylabel('похибка курсу, °')
plt.show()

Що «бачить» фільтр: оцінка зсуву гіроскопа в часі. Для моно і PnP вона сходиться до справжнього значення. Для 3D-3D — ні: фільтр вважає VO незміщеною, тож її власну систематичну похибку курсу (розділ 10) він «списує» на гіроскоп. **Чий це зсув — неспостережувано**, доки немає третього, незалежного джерела.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.axhline(FUS['true_bias_dps'], color=COLORS['GT'], ls='--', lw=2, label=f"справжній зсув {FUS['true_bias_dps']} °/с")
for m in METHODS:
    ax.plot(TIMES - TIMES[0], np.degrees(FUS['bias'][m]), color=COLORS[m], lw=2, label=f'оцінка KF з {m}')
for lo, hi in zip(*[np.flatnonzero(np.diff(np.r_[0, SENS['still'].astype(int), 0]) == s) for s in (1, -1)]):
    ax.axvspan(TIMES[lo] - TIMES[0], TIMES[hi] - TIMES[0], color='#d0d0d0', alpha=0.7, lw=0)
ax.set_xlabel('час, с (сіре — стоїмо, працює ZARU)'); ax.set_ylabel('зсув гіроскопа, °/с'); ax.grid(alpha=0.25)
ax.legend(loc='upper right', fontsize=9)
plt.show()

### 12.4 · Калібрування на старті змінює все
Найпростіше третє джерело — постояти кілька секунд перед стартом (ZARU): середнє показів гіроскопа і є його зсув. Порівняймо три сценарії.

In [ ]:
cols = ['колеса + гіроскоп', '2D-2D + колеса + гіроскоп', '3D-2D', '3D-2D + гіроскоп', '3D-3D', '3D-3D + гіроскоп']
SEEDS = range(10)          # калібрування — випадкова вибірка, тому усереднюємо по 10 реалізаціях шуму
print(f'Похибка курсу в кінці, °: середнє ± σ по {len(SEEDS)} реалізаціях шуму\n')
print(f"{'калібрування':<14}" + ''.join(f'{c:>14}' for c in ['гіроскоп', 'моно+KF', '3D-2D', '3D-2D+KF', '3D-3D', '3D-3D+KF']))
for cs in [0, 5, 30]:
    runs = [fusion_experiment(calib_s=cs, seed=sd)['stats'] for sd in SEEDS]
    cells = [np.array([r[c]['yaw_end'] for r in runs]) for c in cols]
    print(f"{f'{cs} с':<14}" + ''.join(f'{v.mean():>8.2f} ±{v.std():>4.1f}' for v in cells))

Як читати таблицю:
- **0 с** — некалібрований гіроскоп сам «відвертає» на градуси за хвилину. Камера його рятує, якщо VO незміщена (моно, PnP). Зміщену 3D-3D фільтр виправляє лише частково — її похибку він приймає за зсув гіроскопа.
- **5 с** — навіть грубе калібрування різко змінює картину: тепер гіроскоп тримає курс, і 3D-3D з ним у рази краща за себе без гіроскопа.
- **30 с** — добре калібрований гіроскоп сам тримає курс на рівні злиття: на 80 с камера для курсу вже майже нічого не додає. Вона потрібна для масштабу й позиції та для довгих місій, де зсув гіроскопа повільно «пливе» (`BIAS_RW_DPS`).

### 12.5 · Інтерактив
Повзунки: похибка колеса, зсув і шум гіроскопа, тривалість калібрування, довіра до VO.

In [ ]:
from panels import launch_fusion
launch_fusion(fusion_experiment)

## Висновки до слайда «Який метод коли?»
- **3D-2D (PnP)** — найточніший із методів, що не підглядають у GT: метричний масштаб від стерео плюс похибка в пікселях. Саме тому він лежить в основі ORB-SLAM, VINS тощо. «Готова 3D-карта» зі слайда в стерео-VO — це просто тріангуляція попереднього кадру.
- **3D-3D** має метричний масштаб, але погано розділяє поворот і бічний зсув — систематичні похибки стерео перетворюються на зміщення курсу. На практиці його рятують зваженням за коваріацією точок або уточненням через PnP/BA; для лідара (точна глибина) 3D-3D/ICP — основний метод.
- **2D-2D** оцінює обертання не гірше за стерео (він бачить і далекі точки — див. r, °/100м), але без зовнішнього джерела масштабу траєкторія в метрах неправильна. Тому монокуляр у польоті — завжди з IMU, висотоміром чи відомим об'єктом (слайд «Як вирішують проблему масштабу на практиці?»).
- **Дешеві сенсори закривають слабкі місця VO:** колеса дають монокуляру масштаб, гіроскоп — курс. Але фільтр Калмана вірить, що кожне джерело незміщене: без незалежного спостереження (калібрування на старті, ZARU на зупинках) систематичну похибку VO він «спише» на гіроскоп.
- Жоден із методів не позбавлений дрейфу — похибка росте з пройденим шляхом. Без абсолютних вимірювань (GPS, карта, замикання петлі) це не лікується. Це мотивація для loop closure і pose graph у demo3.

## Для самостійного дослідження (необов'язково)
1. **3D-3D зі зваженням:** замініть фіксований поріг 0,3 м на `k·σ_Z` (σ_Z = Z²/(f·B)·0,5 px) і зважте Kabsch вагами `1/σ_Z²`. Наскільки це допомогло? Чому не до рівня PnP? (Підказка: шум анізотропний — він уздовж променя.)
2. **Масштаб для монокуляра без GT:** використайте висоту камери над дорогою (у KITTI ≈ 1,65 м). Тріангулюйте точки дороги в нижній частині кадру, оцініть висоту в «одиницях» і відмасштабуйте кожен крок. Порівняйте ATE з версією «масштаб із GT».
3. **Межа глибини:** поставте `MAX_DEPTH_B = 100` і `200` (і видаліть кеш). Обидва стерео-методи покращаться, 3D-3D — найсильніше. Чому далекі точки, хоч і з шумною глибиною, допомагають? Порівняйте з тим, як ORB-SLAM2 використовує «далекі» стерео-точки лише як монокулярні (для обертання).
4. **Проковзування коліс:** додайте в `simulate_sensors` епізоди, коли колеса крутяться швидше, ніж рухається машина (наприклад, ×1,3 на 2 с). Як це видно на траєкторії «моно + колеса»? Запропонуйте перевірку узгодженості колеса і стерео-VO, яка ловить проковзування.
5. **ZARU без коліс:** для дрона коліс немає. Як визначити «стоїмо» лише з камери та акселерометра? Що станеться з оцінкою зсуву, якщо ZARU спрацює під час повільного повороту?
6. Додатково: як зміняться метрики, якщо брати кожен 2-й кадр (змініть крок у `run_all`)? Який метод деградує найшвидше і чому?